# German Replication Diagnostic

Run this separate notebook to check whether the executed exhibit files are present and internally consistent. It does not modify the main replication notebook or any project data files.


## Run this one cell

The cell below performs the complete diagnostic. You do not need to edit any code or compare results with outside numbers.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

EXPECTED_FILES = [
    'germany_crash_results.xlsx',
    'germany_crash_test.xlsx',
    'germany_footnote15_regressions.xlsx',
    'germany_event_time_paths.xlsx',
    'germany_table1_paper_style.xlsx',
    'germany_table3_main.xlsx',
    'germany_replication_checks.xlsx',
    'germany_event_time_paths.png',
]

def find_project_dir():
    cwd = Path.cwd()
    candidates = [cwd, *cwd.parents]
    for base in [cwd, *cwd.parents]:
        try:
            candidates.extend(item for item in base.iterdir() if item.is_dir())
        except OSError:
            pass
    scored = []
    for candidate in dict.fromkeys(candidates):
        if candidate.is_dir():
            score = sum((candidate / name).exists() for name in EXPECTED_FILES)
            if score:
                scored.append((score, candidate))
    if not scored:
        raise FileNotFoundError('Could not find the folder containing the executed exhibit files.')
    return sorted(scored, key=lambda item: (-item[0], str(item[1])))[0][1]

PROJECT_DIR = find_project_dir()
print(f'Project folder: {PROJECT_DIR}')

print('\n1. Required files')
missing = []
for filename in EXPECTED_FILES:
    exists = (PROJECT_DIR / filename).exists()
    print(('FOUND' if exists else 'MISSING') + f': {filename}')
    if not exists:
        missing.append(filename)

def read_excel(filename):
    path = PROJECT_DIR / filename
    return pd.read_excel(path) if path.exists() else None

crash_results = read_excel('germany_crash_results.xlsx')
crash_test = read_excel('germany_crash_test.xlsx')
footnote15 = read_excel('germany_footnote15_regressions.xlsx')
event_time = read_excel('germany_event_time_paths.xlsx')
replication_checks = read_excel('germany_replication_checks.xlsx')

print('\n2. Exhibit summaries')
if crash_results is not None:
    print('Crash-results rows:', len(crash_results))
    display(crash_results)

if crash_test is not None:
    print('Crash-test rows:', len(crash_test))
    if {'Observations', 'Crashes'}.issubset(crash_test.columns):
        check = crash_test[['Comparison', 'Observations', 'Crashes']].copy()
        check['Calculated crash probability (%)'] = check['Crashes'] / check['Observations'] * 100
        display(check)

if footnote15 is not None:
    print('Footnote-15 regression rows:', len(footnote15))

if event_time is not None:
    print('Event-time rows:', len(event_time))
    if 'event_month' in event_time.columns:
        print('Event month 0 rows:', int((event_time['event_month'] == 0).sum()))

print('\n3. Replication-check exhibit')
if replication_checks is not None:
    display(replication_checks)

print('\n4. Conclusion')
if missing:
    print('Some required files are missing. See the list above.')
else:
    print('All required exhibit files are present.')
print('The diagnostic reports the values in your actual files; it does not label them wrong using outside checkpoints.')
print('The original germany_gsy_replication.ipynb was not modified.')
